# Run `amir/score.py` in Colab

Open this notebook in Google Colab, then choose `Runtime -> Change runtime type` and select a GPU runtime.

This workflow is wired for the `amir/transcripts` corpus. GPU is supported through PyTorch + Transformers. TPU runtimes can still open the notebook, but `score.py` currently falls back to CPU unless you add `torch-xla` support.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Ngafney/garda-spring26.git"
REPO_DIR = Path("/content/garda-spring26") if IN_COLAB else Path.cwd()
USE_GOOGLE_DRIVE = False
GOOGLE_DRIVE_REPO_DIR = Path("/content/drive/MyDrive/garda-spring26")

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = GOOGLE_DRIVE_REPO_DIR

if IN_COLAB and not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(f"Working directory: {REPO_DIR}")

In [ ]:
%pip install -q -r requirements.txt tomli

In [ ]:
import os
import sys
import torch

print("Python:", sys.version.split()[0])
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
if os.environ.get("COLAB_TPU_ADDR"):
    print("TPU runtime detected. score.py will still use CPU unless torch-xla support is added.")

## Parameters

Set `LIMIT = None` for the full corpus. There are about 9,951 transcript files in `amir/transcripts`, so a full run can take a long time even on Colab GPU.

Use `PATTERN` to score a subset first, for example `"amazon"` or `"nvidia"`.

In [ ]:
TRANSCRIPTS_DIR = REPO_DIR / "amir" / "transcripts"
OUTPUT_DIR = REPO_DIR / "amir" / "outputs"
METADATA_CSV = OUTPUT_DIR / "transcript_metadata.csv"
OUTPUT_CSV = OUTPUT_DIR / "transcript_scores.csv"
SCORE_DB = OUTPUT_DIR / "score_cache.sqlite"

LIMIT = 25
PATTERN = None
FORCE = True
VERBOSE = 1

print(TRANSCRIPTS_DIR)
print(OUTPUT_CSV)

In [ ]:
import shlex
import subprocess
import sys

cmd = [
    sys.executable,
    "amir/score.py",
    "--transcripts-dir",
    str(TRANSCRIPTS_DIR),
    "--metadata-csv",
    str(METADATA_CSV),
    "--output-csv",
    str(OUTPUT_CSV),
    "--score-db",
    str(SCORE_DB),
    "--rebuild-metadata",
]

if LIMIT is not None:
    cmd += ["--limit", str(LIMIT)]
if PATTERN:
    cmd += ["--pattern", PATTERN]
if FORCE:
    cmd.append("--force")
cmd += ["-v"] * VERBOSE

print("Running:")
print(" ".join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, check=True)

In [ ]:
import pandas as pd

scores = pd.read_csv(OUTPUT_CSV)
print(f"Rows written: {len(scores):,}")
display(scores.head())